In [ ]:
import healpy as hp
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas
from talltable import PixelQuery
from astropy.table import Table
import pandas as pd
import pickle
from pathlib import Path
from robusta_hmf import Robusta
import matplotlib.animation as animation
import time
import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
# from simbad (https://simbad.u-strasbg.fr/simbad/sim-id?Ident=rosette+nebula&NbIdent=1&Radius=2&Radius.unit=arcmin&submit=submit+id)
ra = 98.11150
dec = 4.79364

In [ ]:
def kt_imshow(image, a = None, b = None, cmap = 'grey_r'):
    u, v = np.nanpercentile(image, [2.5, 97.5])
    if a == None:
        a = ((u + v) / 2 ) - (5/4) * (v - u)
    if b == None:
        b = ((u + v) / 2 ) + (5/4) * (v - u)
    return plt.imshow(image, origin = 'lower', interpolation = 'nearest', vmin = a, vmax = b, cmap = cmap)

def weighted_average(g, val_col, weight_col):
    """
    bug, this doesn't deal with nans correctly
    """
    denominator = np.nansum(1. / g[weight_col])
    numerator = np.nansum(g[val_col] / g[weight_col])
    return  pd.Series({'intensity': numerator / denominator , "intensity_var": 1. / denominator})

# DATA, PIVOT, IMAGE(?) 4 ONE SOURCE

def get_xy_from_int(ii, bits):
    """
    this implements a z-order decode appropriate for hierarchical pixel numbering
    """
    return (sum((ii >> 2*b & 1) << b for b in range(bits)),
            sum((ii >> 2*b+1 & 1) << b for b in range(bits)))

def spherex_wavelength_bin_edges():
    if not hasattr(spherex_wavelength_bin_edges, 'bin_edges'):
        spherex_waves = Table.read('~/Downloads/spectral_channels_spx_cal-sch-v1-2026-106.fits.gz', format='fits')
        spherex_wavelength_bin_edges.bin_edges = 0.5 * (spherex_waves['WL_MAX'][:-1] + spherex_waves['WL_MIN'][1:])
        spherex_wavelength_bin_edges.waves = 0.5 * (spherex_waves['WL_MAX'] + spherex_waves['WL_MIN'])
    return spherex_wavelength_bin_edges.bin_edges, spherex_wavelength_bin_edges.waves

def get_big_pixel_for_source(ra, dec, big_level = 13):
    nside = 2**big_level
    resol = hp.nside2resol(nside, arcmin=True) * 60
    return int(hp.ang2pix(nside, ra, dec, nest=True, lonlat=True))

def get_big_pixels_for_source(ra, dec, big_level = 13):
    """
    default radius set to be about 15 arcseconds
    BUG: THIS SHOULD USE A RADIUS THAT DEPENDS ON BIG_LEVEL AND INCLUSIVE = FALSE
    """
    radius = np.sqrt(hp.pixelfunc.nside2pixarea(2 ** big_level, degrees=False))
    return hp.query_disc(2 ** big_level, hp.pixelfunc.ang2vec(ra, dec, lonlat=True), 
                         radius, inclusive=False, nest=True)

def image_for_one_big_pixel(ra, dec, ipixbig, big_level=13, grid_level=16, verbose=True):
    # QUERYYYYYY
    path = Path(r"C:\Users\katie\saved queries") / f"ipix{ipixbig}_biglevel{big_level}.pkl"
    df_path = Path(r"C:\Users\katie\saved queries") / f"ipix{ipixbig}_biglevel{big_level}_grid_level{grid_level}_df.pkl"
    pivot_path = Path(r"C:\Users\katie\saved queries") / f"ipix{ipixbig}_biglevel{big_level}_grid_level{grid_level}_pivot.pkl"
    image_path = Path(r"C:\Users\katie\saved queries") / f"ipix{ipixbig}_biglevel{big_level}_grid_level{grid_level}_image.pkl"
    if path.exists():
        df = pd.read_pickle(path)
        if verbose:
            print(f"query has been run before so using saved info")
    else:
        table = PixelQuery(web=True).ipix([ipixbig, ], big_level).with_wavelengths().execute()
        df = table.to_pandas()
        del table
        path.parent.mkdir(exist_ok=True)
        df.to_pickle(path)
    if verbose:
        print(f"image_for_one_source(): got {len(df)} measurements")
    
    bin_edges_102, waves102 = spherex_wavelength_bin_edges()
    nlam = len(bin_edges_102) + 1
    if df_path.exists():
        df = pd.read_pickle(df_path)
    else:
        ras, decs = hp.pix2ang(2 ** 22, df['hphigh'], nest=True, lonlat=True)    # magic 22 (needs to be synchronized to the talltable) 
        ipixsmed = hp.ang2pix(2 ** grid_level, ras, decs, nest=True, lonlat=True)
        rasmed, decsmed = hp.pix2ang(2 ** grid_level, ipixsmed, nest=True, lonlat=True)
    
        #GET JLAM102
        bin_edges_102, waves102 = spherex_wavelength_bin_edges()
        nlam = len(bin_edges_102) + 1
        wavelengths = df['wavelength']
        jlam102 = np.digitize(wavelengths, bin_edges_102)
        if verbose:
            print(f"image_for_one_source(): got {len(jlam102)} measurements")
    
        # ADD IPIXSmed, JLAM102, RASmed, DECSmed, WAVES102 TO DF 
        df['ipixmed'] = ipixsmed
        df['ramed'] = rasmed
        df['decmed'] = decsmed
        df['jlam102'] = jlam102
        df['wave102'] = waves102[jlam102]
        
        ### COME BACK TO THIS OR DELETE
        dir1 = [ras, decs]                #from ipix22
        dir2 = [ra, dec]
        thetas = hp.rotator.angdist(dir1, dir2, lonlat=True)
        df['theta'] = thetas * 3600       #degrees to arcseconds
        df['flux_circle'] = df['theta'] <= 5
        df['bkg'] = (df['theta'] > 5) & (df['theta'] <= 10)
        df_path.parent.mkdir(exist_ok=True)
        df.to_pickle(df_path)

    #use pandas to create the relevant pivot table
    if pivot_path.exists():
        pivot = pd.read_pickle(pivot_path)
    else:
        pivot = df.groupby(['ipixmed', 'jlam102']).apply(
                            lambda g: weighted_average(g, 'flux', 'variance')).reset_index()
        rasmed, decsmed = hp.pix2ang(2**grid_level, pivot['ipixmed'], nest=True, lonlat=True)
        pivot['ramed'] = rasmed
        pivot['decmed'] = decsmed
        pivot['ipixbig'] = ipixbig
        pivot_path.parent.mkdir(exist_ok=True)
        pivot.to_pickle(pivot_path)
    
    #laboriously create a 3 dimensional image; maybe brittle
    if image_path.exists():
        image, image_var, image_wcs = pd.read_pickle(image_path)
    else:
        nx = ny = 2 ** (grid_level - big_level)
        yy, xx = get_xy_from_int(pivot['ipixmed'].to_numpy() - (ipixbig << 2 * (grid_level - big_level)), grid_level - big_level) 
        pivot['xxmed'], pivot['yymed'] = xx, yy
        image = np.zeros((nx, ny, nlam)) + np.nan
        image[xx, yy, pivot['jlam102'].to_numpy()] = pivot['intensity']
        image_var = np.zeros((nx, ny, nlam)) + np.nan
        image_var[xx, yy, pivot['jlam102'].to_numpy()] = pivot['intensity_var']
        image_wcs = np.zeros((nx, ny, 2)) + np.nan
        image_wcs[xx, yy, 0] = pivot['ramed']    #bad code
        image_wcs[xx, yy, 1] = pivot['decmed']   #bad code
        image_path.parent.mkdir(exist_ok=True)
        pd.to_pickle((image, image_var, image_wcs), image_path)
    return df, pivot, image, image_var, image_wcs

In [ ]:
big_level = 7 #getting a patch for hogg
ipixbig = get_big_pixel_for_source(ra, dec, big_level = big_level)
df, pivot, image, image_var, image_wcs = image_for_one_big_pixel(ra, dec, ipixbig, big_level = big_level, 
                                                                 grid_level = 16, verbose = True)

In [ ]:
jj = 62
mask = pivot['jlam102'] == jj
rasmedj = image_wcs[:, :, 0].flatten()
decsmedj = image_wcs[:, :, 1].flatten()
intensityj = image[:, :, jj].flatten()
plt.scatter(rasmedj, decsmedj, c = intensityj, marker = '.', s = 0.5)
plt.colorbar()
plt.xlabel("Right ascension [deg]")
plt.ylabel("Declination [deg]")

In [ ]:
plt.imshow(image[:, :, jj])

In [ ]:
kt_imshow(image[:, :, jj])
#plt.savefig('data_image.png', bbox_inches='tight')
fig, ax = plt.subplots()
ims = []
for i in range(102):
    im = kt_imshow(image[:, :, i])
    ims.append([im])

ani = animation.ArtistAnimation(fig, ims, interval=100, blit=True,
                                repeat_delay=1000)
ani.save("data_image.gif", writer="pillow")

In [ ]:
# get appropriate jlam
filtered = [x for x in df['wavelength'] if 3.30111 > x > 3.300000]
smallest = min(filtered)
filtered_df = df.loc[df['wavelength'] == smallest]

In [ ]:
data = image.reshape(-1, 102).copy()
#image_var gives variance, not inverse variance
inv_image_var = 1 / image_var
ivar = inv_image_var.reshape(-1, 102)
bad_pixels = ~ np.isfinite(data)
data[bad_pixels] = (np.nanmean(data, axis = 1)[:, None] * np.ones(data.shape[1])[None, :])[bad_pixels]
bad = ~np.isfinite(data)
data[bad] = 0.
ivar[bad] = 0.
bad = ~np.isfinite(ivar)
data[bad] = 0.
ivar[bad] = 0.

In [ ]:
# we need at least one pixel at each wavelength and at least one wavelength at each pixel
# this is a hack
TINY = 1e-6 * np.median(ivar[ivar > 0])
A=np.max(ivar, axis=0)
bad = A < TINY
print(np.sum(bad))
data[:, bad] = 0
ivar[:, bad] = TINY
B=np.max(ivar, axis=1)
bad = B < TINY
data[bad, :] = 0
ivar[bad, :] = TINY
#set robust parameters to 1

In [ ]:
NN, V = data.shape
rr = np.random.uniform(size = NN)
idxA = rr < 0.5
idxB = rr >= 0.5

In [ ]:
K = 12
model = Robusta(rank = K, robust = True, robust_scale = 1.0)
state, loss_history = model.fit(data, ivar, max_iter = 1000)

basis = state.G          # (M, K) shared basis vectors
coeffs = state.A         # (N, K) per-row coefficients
reconstruction = model.synthesize()   # A @ G.T, shape (N, M)

In [ ]:
robusta_image = reconstruction.reshape(512,512,102)

In [ ]:
kt_imshow(robusta_image[:,:,jj])
plt.savefig('robusta_image.png', bbox_inches='tight')

In [ ]:
bad_image = bad_pixels.reshape(512,512,102)
replaced_nans_image = np.where(bad_image, robusta_image, image)
kt_imshow(replaced_nans_image[:,:,jj])
plt.savefig('patched_image.png', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots()
ims = []
for i in range(102):
    im = kt_imshow(replaced_nans_image[:, :, i])
    ims.append([im])

ani = animation.ArtistAnimation(fig, ims, interval=100, blit=True,
                                repeat_delay=1000)
ani.save("patched_image.gif", writer="pillow")

In [ ]:
#from https://github.com/nanamiller/EmissionLineStars/blob/main/notebooks/setting_hyperparameters.ipynb
Ks = np.array([4, 8, 16])
results_A = []
for K in Ks:
    start = time.perf_counter()
    modelA = Robusta(rank=K, robust=True, robust_scale=1, robust_nu=1)
    modelA.fit(data[idxA, :], ivar[idxA, :], max_iter=10000)
    
    state_A, _ = modelA.infer(data[idxB, :], ivar[idxB, :], max_iter=10000)
    synth_B = modelA.synthesize(state_A)
    residual_A = data[idxB, :] - synth_B
    mad_A = np.nanmedian(np.abs(residual_A))
    rms_A = np.sqrt(np.nanmean(residual_A ** 2))

    end = time.perf_counter()
    print("total time:", (end - start)/60, "minutes")
    
    results_A.append({"K": K, "MAD": mad_A, "rms": rms_A, "residual": residual_A})
    
    print(f"K={K} and MAD={mad_A:.5f}")

In [ ]:
#from https://github.com/nanamiller/EmissionLineStars/blob/main/notebooks/setting_hyperparameters.ipynb
Ks = [4, 8, 16]
results_B = []
for K in Ks:
    start = time.perf_counter()
    modelB = Robusta(rank=K, robust=True, robust_scale=1, robust_nu=1)
    modelB.fit(data[idxB, :], ivar[idxB, :], max_iter=10000)
    
    state_B, _ = modelB.infer(data[idxA, :], ivar[idxA, :], max_iter=10000)
    synth_A = modelB.synthesize(state_B)
    residual_B = data[idxA] - synth_A
    mad_B = np.nanmedian(np.abs(residual_B))
    rms_B = np.sqrt(np.nanmean(residual_B ** 2))

    end = time.perf_counter()
    print("total time:", (end - start)/60, "minutes")
    
    results_B.append({"K": K, "MAD": mad_B, "rms": rms_B, "residual": residual_B})
    
    print(f"K={K} and MAD={mad_B:.5f}")

In [ ]:
rms_A = [results_A[0]['rms'], results_A[1]['rms'], results_A[2]['rms']]
mads_A = [results_A[0]['MAD'], results_A[1]['MAD'], results_A[2]['MAD']]
plt.scatter(rms_A, Ks)
plt.scatter(mads_A, Ks)

In [ ]:
plt.scatter(mads_A, Ks)
plt.scatter(mads_B, Ks)

In [ ]:
rms_B = [results_B[0]['rms'], results_B[1]['rms'], results_B[2]['rms']]
mads_B = [results_B[0]['MAD'], results_B[1]['MAD'], results_B[2]['MAD']]
plt.scatter(rms_B, Ks)
plt.scatter(mads_B, Ks)

In [ ]:
foo = Robusta.fit